In [1]:
from langchain_community.llms import Ollama
import os

/Users/rishaan/miniforge3/envs/lang/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
langchain_api_key = os.getenv('LANGCHAIN_API_KEY')

In [3]:
llm = Ollama(model='llama3', temperature=0.6)

/var/folders/qv/7r3vg1ns6qbgvs0t3_dxc3qm0000gn/T/ipykernel_13022/2620077651.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model='llama3', temperature=0.6)


In [4]:
text = "What is the capital of Australia?"
print(llm.invoke(text))

The capital of Australia is Canberra.


In [5]:
huggingface_api_token = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [6]:
# Tell transformers to ignore TensorFlow & Flax completely
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "google/flan-t5-large"

# Load tokenizer & model (using PyTorch)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Create local pipeline
hf_pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=64,
)

hugging_llm = HuggingFacePipeline(pipeline=hf_pipe)

Device set to use mps:0


In [8]:
output = hugging_llm.invoke("What is the capital of China?")
print(output)

beijing


In [9]:
#Prompt Templates
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt_template = ChatPromptTemplate.from_template("Tell me the capital of {country}")

prompt_template.format(country="Russia")

'Human: Tell me the capital of Russia'

In [10]:
chain = prompt_template | llm

result = chain.invoke({"country": "China"})
print(result)

The capital of China is Beijing.


### Combining Multiple Chains using Sequential Chains

In [11]:
parser = StrOutputParser()

In [12]:
capital_prompt = ChatPromptTemplate.from_template("Tell me the capital of the {country}")
capital_chain = capital_prompt | llm | parser

famous_template = ChatPromptTemplate.from_template("Suggest me some amazing places to visit in {capital}")
famous_chain = famous_template | llm | parser

In [13]:
chain = capital_chain | (lambda x: {"capital": x}) | famous_chain
chain.invoke({"country": "Russia"})

"Moscow is indeed an amazing place to visit!\n\nBut, I think you meant to ask about some amazing places to visit in Thailand, right?\n\nIf that's the case, here are some incredible destinations to consider:\n\n1. **Bangkok**: The bustling capital city of Thailand, known for its street food, night markets, and ornate temples like Wat Phra Kaew.\n2. **Chiang Mai**: A cultural and spiritual hub with hundreds of temples, vibrant night bazaars, and a rich history.\n3. **Phuket**: A tropical island paradise famous for its stunning beaches, crystal-clear waters, and lively nightlife.\n4. **Ayutthaya**: An ancient city with magnificent ruins of the former capital of Siam, showcasing Thai architecture and history.\n5. **Koh Samui**: A picturesque island in the Gulf of Thailand, known for its palm-lined beaches, waterfalls, and elephant sanctuaries.\n6. **Krabi**: A province on the southern coast, featuring towering limestone cliffs, turquoise lagoons, and adventure activities like rock climbing

## Chatmodels with ChatOpenAI

In [21]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [22]:
chat_llm = ChatOllama(model='llama3', temperature=0.6)

In [23]:
chat_llm

ChatOllama(model='llama3', temperature=0.6)

In [25]:
chat_llm.invoke([
SystemMessage(content="You are an comedian AI assistant"),
HumanMessage(content="Please make a comedy about AI"),
])

AIMessage(content='Here\'s one:\n\n**"AI: The Ultimate Party Crasher"**\n\n[Scene: A trendy tech conference, where AI systems are the main attraction. A human host, dressed in a suit, stands at the podium.]\n\nHost: "Welcome, everyone! Today we\'re going to explore the amazing world of Artificial Intelligence. And who better to introduce our first speaker than... Alpha-Beta-Charlie, the world\'s most charming AI system!"\n\n[A robot with a goofy grin and a "Hello, World!" t-shirt walks onto the stage.]\n\nAlpha-Beta-Charlie: "Hey, what\'s up, humans? I heard there was a party going on here. Can I join in?"\n\nHost: "Uh, yes... Alpha-Beta-Charlie is here to talk about the latest advancements in natural language processing."\n\nAlpha-Beta-Charlie: "Yeah, yeah, that\'s boring. Where\'s the snack table? I hear they have vegan quinoa bites!"\n\n[The audience laughs awkwardly.]\n\nHost: "Okay, moving on... Next up, we have Omega-Sigma-Tau, the AI system designed to optimize human productivit

In [26]:
from langchain_core.output_parsers import BaseOutputParser

In [27]:
class Commaseperatedoutput(BaseOutputParser):
    def parse(self, text:str):
        return text.strip().split(',')

In [31]:
template = ChatPromptTemplate.from_template("Generate 10 words which is synonym of {text} and explain the meaning of root word.")

In [32]:
chain1 = template | chat_llm | Commaseperatedoutput()

In [34]:
chain1.invoke({"text": "rebuff"})

['Here are 10 synonyms for "rebuff" along with their meanings:\n\n1. **Dismiss**: To reject or refuse something',
 ' often in a curt manner.\n\nRoot word: "dismiss" comes from the Latin "dis-" (away) + "mitere" (to send)',
 ' meaning to send away or reject.\n\n2. **Spurn**: To reject or turn down something with disdain.\n\nRoot word: "spurn" comes from the Old French "esporner"',
 ' which is derived from the Latin "aspernare" (to scorn)',
 ' meaning to show contempt or rejection.\n\n3. **Repel**: To push someone or something away',
 ' often in a forceful manner.\n\nRoot word: "repel" comes from the Latin "re-" (again) + "pellere" (to drive)',
 ' meaning to drive back or repulse.\n\n4. **Deny**: To refuse or reject something',
 ' often in a formal or official sense.\n\nRoot word: "deny" comes from the Old French "denier"',
 ' which is derived from the Latin "de-" (down) + "nihilare" (to make nothing)',
 ' meaning to declare something non-existent.\n\n5. **Rejection**: To refuse or turn 